In [ ]:
import numpy as np
import pandas as pd
from config import *

In [ ]:
df_edv = read_influenza_ed_visits_prop_data(data_dir, loc_name2abbr)
states = df_edv.columns[3:]
num_states = len(states)
locations = locations.loc[states,]
df_edv[states] = np.round(df_edv[states].mul(pop_per_loc_abbr.loc[states], axis=1) * 1e-3,0)
df_edv

In [ ]:
ensemble_models = ["ExponentialSmoothing_edv_mod_ili", 
                   "LightGBM_edv_mod_ili", 
                   "seasonal_drift_edv_mod_ili",
                   "SIR-EAKF_edv_start_251004"] #"SIR-EAKF_edv_start_250809"]

ensemble_dict = {}
# load models predictions
for model in ensemble_models:
    print("-----------Loading model: {}-----------".format(model))
    df_results = load_pred_result_files(results_dir, model, locations)
    ensemble_dict[model] = df_results

mean_ensemble_name = 'Mean_CU_Ensemble_edv'
weighted_ensemble_name = 'WIS_Weighted_CU_Ensemble_edv'

start_date = pd.to_datetime('2025-11-08', format="%Y-%m-%d")

print("-----------Generating mean ensemble: {}-----------".format(mean_ensemble_name))
generate_mean_ensemble_pred_results(ensemble_dict, start_date, results_dir, mean_ensemble_name)
df_metrics_models = calc_models_pred_fit(df_edv, ensemble_models, season, locations, 
                                  results_dir, figures_dir, alpha_vals, plot=False)
df_weights = generate_pred_weights(ensemble_dict, df_metrics_models, locations, decay_rate=1.0)
print("-----------Generating weighted ensemble: {}-----------".format(weighted_ensemble_name))
generate_weighted_pred_results(ensemble_dict, start_date, df_weights, locations, results_dir, weighted_ensemble_name) 

df_metrics_ensembles = calc_models_pred_fit(df_edv, [mean_ensemble_name,weighted_ensemble_name],season,
                                            locations, results_dir, figures_dir, alpha_vals, plot=False)

df_metrics = pd.concat([df_metrics_models, df_metrics_ensembles], ignore_index=True)
df_metrics.to_csv(results_dir+"/_metrics/metrics_edv_" +ref_date.strftime("%Y-%m-%d") +'.csv',index=False)

In [ ]:
# df_metrics = pd.read_csv(results_dir+"/_metrics/metrics_edv_" +ref_date.strftime("%Y-%m-%d") +'.csv')
# additional_model = ["FluSight-baseline_edv"] 
# df_metrics = df_metrics[~df_metrics['model'].isin(additional_model)]
# df_metrics_additional = calc_models_pred_fit(df_edv, additional_model, season, locations, 
#                                              results_dir, figures_dir, alpha_vals, plot=False)
# df_metrics = pd.concat([df_metrics, df_metrics_additional], ignore_index=True) 
# df_metrics.to_csv(results_dir+"/_metrics/metrics_edv_" +ref_date.strftime("%Y-%m-%d") +'.csv',index=False)

In [ ]:
# models_to_plot = ensemble_models + [mean_ensemble_name,weighted_ensemble_name]
models_to_plot = [mean_ensemble_name,weighted_ensemble_name] 
calc_models_pred_fit(df_edv, models_to_plot, season, locations, 
                     results_dir, figures_dir, alpha_vals, plot=True)